In [ ]:
from graphgen.config_utils import generate_and_save_static_config
from graphgen.runner import batch_sample_and_save
from embedding.pipeline import run_embedding_and_prediction_pipeline
from embedding.plot import plot_tsne, plot_raw
from evaluation.runner import run_evaluation_pipeline
from evaluation.plot import plot_multi_modes
from evaluation.p_values import compute_p_values

from sklearn.metrics import normalized_mutual_info_score as NMI
from sklearn.metrics import adjusted_rand_score as ARI
from pathlib import Path
import numpy as np
from itertools import product

In [ ]:
p_values = [1.0, 0.9, 0.8, 0.7, 0.6, 0.55, 0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2, 0.1]
q = 0.2
pq_list = [(p, q) for p in p_values]
runs_per_pq = 20

K = 3
label_mappings = [
                 {0: 0, 1: 1, 2: 2},
                 {0: 0, 1: 0, 2: 1},
                 {0: 0, 1: 1, 2: 1}
                 ]
clusters_per_time = [3, 2, 2]
n_nodes = 200
connected = False

emb_type_list = [
                {"rep_type": "UASE", "regularized": 0},
                {"rep_type": "ULSE-n1", "regularized": 0.1},
                {"rep_type": "ULSE-n2", "regularized": 0.1},
                ]
metrics = {"NMI": NMI, "ARI": ARI}

pi = np.array([0.3, 0.4, 0.3]) # 1
# pi = np.array([0.4, 0.5, 0.1]) # 2
base_dir = Path("../data/synthetic/1")
emb_dir = Path("../emb/synthetic/1")
true_label_path =  base_dir / "config/node2label.txt"

In [ ]:
# graph sampling
static_labels, dynamic_labels = generate_and_save_static_config(
    n=n_nodes,
    K=K,
    pi=pi,
    label_mappings=label_mappings,
    config_dir= base_dir / "config",
    seed=42,
)
batch_sample_and_save(
        static_labels=static_labels,
        dynamic_node_labels=dynamic_labels,
        output_dir=base_dir,
        pq_list=pq_list,
        runs_per_pq=runs_per_pq,
        base_seed=42,
        connected=False
)

In [ ]:
# embedding and clustering
run_embedding_and_prediction_pipeline(
    base_dir = base_dir,
    pq_list = pq_list,
    runs_per_pq = runs_per_pq,
    clusters_per_time = clusters_per_time,
    emb_size = K,
    random_state = 42,
    emb_type_list = emb_type_list,
    emb_dir = emb_dir
)

In [ ]:
# evaluation
result = run_evaluation_pipeline(
    base_dir = emb_dir,
    pq_list = pq_list,
    runs_per_pq = runs_per_pq,
    n_nodes = n_nodes,
    true_label_path = true_label_path,
    emb_type_list = emb_type_list,
    metrics = metrics,
)

In [ ]:
# accuracy plot
plot_multi_modes(
    pq_list = pq_list,
    base_dir = emb_dir,
    emb_type_list = emb_type_list,
    metrics = metrics,
)

In [ ]:
# compute p values
compute_p_values(
    base_dir=emb_dir,
    emb_type_list=emb_type_list,
    metrics=metrics,
    sample_size=runs_per_pq,
    emb_low_key="UASE_reg0",
    emb_high_key="ULSE-n1_reg0.1",
)
compute_p_values(
    base_dir=emb_dir,
    emb_type_list=emb_type_list,
    metrics=metrics,
    sample_size=runs_per_pq,
    emb_low_key="UASE_reg0",
    emb_high_key="ULSE-n2_reg0.1",
)

In [ ]:
# embedding plot (raw)
method_list = ["USE_UASE", "USE_ULSE-n1", "USE_ULSE-n2", "TemporalCut_fast_normalized_1_0"]
label_path = Path("../data/synthetic/2/config/static_node2label.txt")
for method in method_list:
    for t in [2,3]:
        emb_path = Path("../emb/synthetic_2/synthetic_2_"+method+".emb")
        save_path = Path("./plots/synthetic_2_raw_"+ str(t) +"timesteps_"+method+".png")

        if method=="USE_ULSE-n1":
            plot_raw(
                embedding_path=emb_path,
                label_path=label_path,
                save_path= save_path,
                method_name = method,
                columns=[1,2],
                n_times = t
            )
        else:
            plot_raw(
                embedding_path=emb_path,
                label_path=label_path,
                save_path= save_path,
                method_name = method,
                columns=[0,1],
                n_times = t
            )


In [ ]:
# embedding plot (tsne)
method_list = ["USE_UASE", "USE_ULSE-n1", "USE_ULSE-n2"]
label_path = Path("../data/synthetic/2/config/static_node2label.txt")
for method in method_list:
    emb_path = Path("../emb/synthetic_2/synthetic_2_"+method+".emb")
    save_path = Path("./plots/synthetic_2_tsne_3timesteps_"+method+".png")

    plot_tsne(
        embedding_path=emb_path,
        label_path=label_path,
        save_path= save_path,
        method_name = method,
    )